# Step 4c — Learned Ensemble Weights
DINOv2 + DINOv3 + SAM · score-level fusion · learned softmax weights · SPair-71k / PF-Pascal / PF-Willow

In [1]:
# Cell 0 — Mount Drive
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE_ROOT = '/content/drive/MyDrive/semantic_correspondence'
RESULTS_DIR = f'{DRIVE_ROOT}/results/step4/ensemble'
os.makedirs(RESULTS_DIR, exist_ok=True)
print('Drive mounted.')

Mounted at /content/drive
Drive mounted.


In [9]:
# Cell 0b — Copy SPair-71k from Drive to local SSD
import shutil, os, time, concurrent.futures

LOCAL_SPAIR = '/content/spair71k'
DRIVE_SPAIR = f'{DRIVE_ROOT}/datasets/SPair-71k'

def _count(path):
    return sum(len(fs) for _, _, fs in os.walk(path))

needs_copy = True
if os.path.exists(LOCAL_SPAIR):
    lc, dc = _count(LOCAL_SPAIR), _count(DRIVE_SPAIR)
    if lc == dc:
        needs_copy = False
        print(f'Local SPair-71k complete ({lc:,} files). Skipping copy.')
    else:
        print(f'Incomplete copy ({lc:,}/{dc:,} files). Deleting and re-copying...')
        shutil.rmtree(LOCAL_SPAIR)

if needs_copy:
    pairs = []
    for root, dirs, files in os.walk(DRIVE_SPAIR):
        local_root = root.replace(DRIVE_SPAIR, LOCAL_SPAIR)
        os.makedirs(local_root, exist_ok=True)
        for fname in files:
            pairs.append((os.path.join(root, fname), os.path.join(local_root, fname)))
    print(f'Copying {len(pairs):,} files with 16 threads...')
    t0 = time.time()
    def _copy(p): shutil.copy2(p[0], p[1])
    with concurrent.futures.ThreadPoolExecutor(max_workers=16) as ex:
        list(ex.map(_copy, pairs))
    print(f'Done in {time.time()-t0:.1f}s')
print(f'Dataset path: {LOCAL_SPAIR}')

Incomplete copy (23,032/76,372 files). Deleting and re-copying...
Copying 76,372 files with 16 threads...
Done in 3408.3s
Dataset path: /content/spair71k


In [10]:
# Cell 1  Install packages + clone repo for model architectures
!pip install -q timm scikit-learn pandas segment-anything

import os, subprocess
REPO_PATH = '/content/semantic_correspondence'
if not os.path.exists(REPO_PATH):
    # Replace URL with your actual repo URL
    subprocess.run(['git', 'clone', '--depth', '1', '-b', 'local-notebooks',
                    'https://github.com/Hesam-AHT/Semantic-Correspondence-with-Visual-Foundation-Models.git',
                    REPO_PATH], check=False)
    print('Repo cloned.')
else:
    print('Repo already present.')

import sys
if REPO_PATH not in sys.path:
    sys.path.insert(0, REPO_PATH)
print('sys.path updated.')

Repo already present.
sys.path updated.


In [11]:
# Cell -- Download pretrained weights (guarded)
import subprocess

DINOV2_W = f'{DRIVE_ROOT}/weights/dinov2_vitb14_pretrain.pth'
if not os.path.exists(DINOV2_W):
    print('Downloading DINOv2 weights...')
    subprocess.run(['wget', '-q', '-O', DINOV2_W,
        'https://dl.fbaipublicfiles.com/dinov2/dinov2_vitb14/dinov2_vitb14_pretrain.pth'])
    print('DINOv2 weights saved.')
else:
    print('DINOv2 weights already present.')

SAM_W = f'{DRIVE_ROOT}/weights/sam_vit_b.pth'
if not os.path.exists(SAM_W):
    print('Downloading SAM ViT-B weights...')
    subprocess.run(['wget', '-q', '-O', SAM_W,
        'https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth'])
    print('SAM weights saved.')
else:
    print('SAM weights already present.')

DINOV3_W = f'{DRIVE_ROOT}/weights/dinov3_vitb16_pretrain.pth'
if not os.path.exists(DINOV3_W):
    print('Downloading DINOv3 weights...')
    subprocess.run(['wget', '-q', '-O', DINOV3_W,
        'https://dinov3.llamameta.net/dinov3_vitb16/dinov3_vitb16_pretrain_lvd1689m-73cec8be.pth?Policy=eyJTdGF0ZW1lbnQiOlt7InVuaXF1ZV9oYXNoIjoiaXNuZXB2NWJyMmpvcXM4N3Y2dTlrNDFlIiwiUmVzb3VyY2UiOiJodHRwczpcL1wvZGlub3YzLmxsYW1hbWV0YS5uZXRcLyoiLCJDb25kaXRpb24iOnsiRGF0ZUxlc3NUaGFuIjp7IkFXUzpFcG9jaFRpbWUiOjE3Nzk0ODQ1NTF9fX1dfQ__&Signature=s4mL8ILUFqU0eE1KBha8ryjjPJzobeZNek7-G~skvFiuTaDOvBw47eWenOENnHNWN8iPISuCsj5zQJt7pZRGMDEbbq1D3yTQOuNjDFRvX95Xx6wtNYb7TxkP3xQJD-OEWQu3L-5x-wHVW-m-nLTmKyMN1v~QTFoWm6OWixvAz3qVpyK4Ho2xeShlUpCdX19vONCk2EPGwj0akN4AgI5sAaLz9VmY4VRXygrEARhMhhGqBs0FuaKZXdkggMOW~gH3NuEia3BJn84jQLKPTbu1HZa8flz0NQ5~ZvetXOFS-SbrpaIEuwZiLTVdP1djkV8tZsNugBue-savfiLrnJpgvw__&Key-Pair-Id=K15QRJLYKIFSLZ&Download-Request-ID=914567197795521'])
    print('DINOv3 weights saved.')
else:
    print('DINOv3 weights already present.')

DINOv2 weights already present.
SAM weights already present.
DINOv3 weights already present.


In [12]:
# Cell 2 — Imports + config
import torch, numpy as np, json, time, os, math
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
from PIL import Image
from torchvision import transforms
from torch.utils.data import Dataset

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

LOCAL_SPAIR   = '/content/spair71k'
SPAIR_BASE    = f'{LOCAL_SPAIR}/SPair-71k'
PAIR_ANN_PATH = f'{SPAIR_BASE}/PairAnnotation'
LAYOUT_PATH   = f'{SPAIR_BASE}/Layout'
IMAGE_PATH    = f'{SPAIR_BASE}/JPEGImages'
DATASET_SIZE  = 'large'
PCK_ALPHA     = 0.1
THRESHOLDS    = [0.05, 0.1, 0.2]
DINOV2_W      = f'{DRIVE_ROOT}/weights/dinov2_vitb14_pretrain.pth'
DINOV3_W      = f'{DRIVE_ROOT}/weights/dinov3_vitb16_pretrain.pth'
SAM_W         = f'{DRIVE_ROOT}/weights/sam_vit_b.pth'
FINETUNED_DINOV2 = f'{DRIVE_ROOT}/weights/finetuned/dinov2_best.pth'
FINETUNED_DINOV3 = f'{DRIVE_ROOT}/weights/finetuned/dinov3_best.pth'
SAM_GRID_SIZE = 32

Device: cuda


In [13]:
# Cell 3 — Inline utility functions
class Normalize:
    def __init__(self, image_keys):
        self.image_keys = image_keys
        self.normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    def __call__(self, sample):
        for key in self.image_keys:
            sample[key] /= 255.0
            sample[key] = self.normalize(sample[key])
        return sample

def read_img(path):
    img = np.array(Image.open(path).convert('RGB'))
    return torch.tensor(img.transpose(2, 0, 1).astype(np.float32))

class SPairDataset(Dataset):
    def __init__(self, pair_ann_path, layout_path, image_path, dataset_size, pck_alpha, datatype):
        self.ann_files = open(os.path.join(layout_path, dataset_size, datatype + '.txt')).read().split('\n')
        self.ann_files = self.ann_files[:len(self.ann_files) - 1]
        self.pair_ann_path = pair_ann_path
        self.datatype = datatype
        self.image_path = image_path
        self.transform = Normalize(['src_img', 'trg_img'])
    def __len__(self): return len(self.ann_files)
    def __getitem__(self, idx):
        ann_file = self.ann_files[idx] + '.json'
        with open(os.path.join(self.pair_ann_path, self.datatype, ann_file)) as f:
            ann = json.load(f)
        category = ann['category']
        src_img = read_img(os.path.join(self.image_path, category, ann['src_imname']))
        trg_img = read_img(os.path.join(self.image_path, category, ann['trg_imname']))
        sample = {'src_imname': ann['src_imname'], 'trg_imname': ann['trg_imname'],
                  'src_imsize': src_img.size(), 'trg_imsize': trg_img.size(),
                  'trg_bbox': ann['trg_bndbox'], 'category': category,
                  'src_img': src_img, 'trg_img': trg_img,
                  'src_kps': torch.tensor(ann['src_kps']).float(),
                  'trg_kps': torch.tensor(ann['trg_kps']).float(),
                  'kps_ids': ann['kps_ids']}
        return self.transform(sample)

def extract_dense_features(model, img_tensor):
    with torch.no_grad():
        fd = model.forward_features(img_tensor)
        pt = fd['x_norm_patchtokens']
        B, N, D = pt.shape
        H = W = int(N ** 0.5)
        return pt.reshape(B, H, W, D)

def extract_dense_features_SAM(model, img_tensor, image_size=512):
    with torch.no_grad():
        img_r = F.interpolate(img_tensor, size=(image_size, image_size), mode='bilinear', align_corners=False)
        if image_size != 1024:
            orig_pe = model.image_encoder.pos_embed
            ng = image_size // 16
            pe_r = F.interpolate(orig_pe.permute(0,3,1,2), size=(ng,ng),
                                  mode='bicubic', align_corners=False).permute(0,2,3,1)
            model.image_encoder.pos_embed = nn.Parameter(pe_r, requires_grad=False)
        from torch.cuda.amp import autocast
        with autocast():
            emb = model.image_encoder(img_r)
        if image_size != 1024:
            model.image_encoder.pos_embed = orig_pe
        return emb.permute(0, 2, 3, 1)

def pixel_to_patch_coord(x, y, original_size, patch_size=14, resized_size=518):
    sx = resized_size / original_size[0]
    sy = resized_size / original_size[1]
    px = int(x * sx // patch_size)
    py = int(y * sy // patch_size)
    mx = resized_size // patch_size - 1
    return min(max(px, 0), mx), min(max(py, 0), mx)

def patch_to_pixel_coord(patch_x, patch_y, original_size, patch_size=14, resized_size=518):
    xr = patch_x * patch_size + patch_size / 2
    yr = patch_y * patch_size + patch_size / 2
    return xr * original_size[0] / resized_size, yr * original_size[1] / resized_size

def find_best_match_argmax(s, width):
    idx = s.argmax().item()
    return idx % width, idx // width

def compute_pck_spair71k(pred_points, gt_points, bbox, threshold):
    pred = np.array(pred_points)
    gt   = np.array(gt_points)
    dist = np.sqrt(np.sum((pred - gt) ** 2, axis=1))
    norm = max(bbox[2] - bbox[0], bbox[3] - bbox[1])
    nd   = dist / norm
    mask = nd <= threshold
    return float(np.mean(mask) * 100), mask, nd

class LearnedEnsembleWeights(nn.Module):
    """Learnable per-model weights via softmax over logits.
    Initialized to zeros -> uniform weights at start."""
    def __init__(self, n_models=3):
        super().__init__()
        self.logits = nn.Parameter(torch.zeros(n_models))
    def forward(self):
        return torch.softmax(self.logits, dim=0)

def save_exp_results(per_img, name, thresholds, results_dir):
    stats = {}
    for thr in thresholds:
        pcks = [m['pck_scores'][thr] for m in per_img]
        stats[f'pck@{thr:.2f}'] = {'mean': float(np.mean(pcks)), 'std': float(np.std(pcks))}
        print(f'  PCK@{thr:.2f}: {np.mean(pcks):.2f}% ± {np.std(pcks):.2f}%')
    out = {'name': name, 'n_pairs': len(per_img), 'stats': stats}
    path = os.path.join(results_dir, f'{name}.json')
    with open(path, 'w') as f: json.dump(out, f, indent=2)
    print(f'  Saved -> {path}')
    return stats

print('Utility functions loaded.')

Utility functions loaded.


In [14]:
# Cell 4 — Load all three models
from src.models.dinov2.dinov2.models.vision_transformer import vit_base as dinov2_vit_base
from src.models.segment_anything.segment_anything import sam_model_registry

# DINOv2
dinov2 = dinov2_vit_base(
    img_size=(518, 518), patch_size=14,
    num_register_tokens=0, block_chunks=0, init_values=1.0
).to(device)
if os.path.exists(FINETUNED_DINOV2):
    ckpt = torch.load(FINETUNED_DINOV2, map_location=device)
    dinov2.load_state_dict(ckpt['model_state_dict'], strict=False)
    print('DINOv2 finetuned loaded.')
else:
    ckpt = torch.load(DINOV2_W, map_location=device)
    dinov2.load_state_dict(ckpt, strict=True)
    print('DINOv2 pretrained loaded.')
dinov2.eval()

# DINOv3
try:
    from src.models.dinov3.dinov3.models.vision_transformer import vit_base as dinov3_vit_base
    dinov3 = dinov3_vit_base(
        img_size=(512, 512), patch_size=16,
        n_storage_tokens=4, layerscale_init=1.0e-05, mask_k_bias=True
    ).to(device)
    if os.path.exists(FINETUNED_DINOV3):
        ckpt3 = torch.load(FINETUNED_DINOV3, map_location=device)
        dinov3.load_state_dict(ckpt3['model_state_dict'], strict=False)
        print('DINOv3 finetuned loaded.')
    else:
        ckpt3 = torch.load(DINOV3_W, map_location=device)
        dinov3.load_state_dict(ckpt3, strict=True)
        print('DINOv3 pretrained loaded.')
    dinov3.eval()
    DINOV3_OK = True
except Exception as e:
    print(f'DINOv3 unavailable ({e}). Will use DINOv2 as proxy for model-2.')
    dinov3 = None
    DINOV3_OK = False

# SAM
sam = sam_model_registry['vit_b'](checkpoint=SAM_W).to(device)
sam.eval()
print('SAM loaded.')

DINOv2 finetuned loaded.
DINOv3 pretrained loaded.
SAM loaded.


In [16]:
# Cell 5 — Train ensemble weights on SPair-71k train split
# Features extracted once per pair (cached); gradients flow only through weight_module.
train_ds = SPairDataset(PAIR_ANN_PATH, LAYOUT_PATH, IMAGE_PATH, DATASET_SIZE, PCK_ALPHA, datatype='trn')
val_ds   = SPairDataset(PAIR_ANN_PATH, LAYOUT_PATH, IMAGE_PATH, DATASET_SIZE, PCK_ALPHA, datatype='val')

weight_module = LearnedEnsembleWeights(n_models=3).to(device)
optimizer_w = torch.optim.Adam(weight_module.parameters(), lr=1e-2)

model2 = dinov3 if DINOV3_OK else dinov2
IMG_SIZES   = [518, 512 if DINOV3_OK else 518, 512]
PATCH_SIZES = [14,  16  if DINOV3_OK else 14,  16 ]

models_info = [
    (dinov2, IMG_SIZES[0], PATCH_SIZES[0], False),
    (model2, IMG_SIZES[1], PATCH_SIZES[1], False),
    (sam,    IMG_SIZES[2], PATCH_SIZES[2], True),
]

TRAIN_EPOCHS     = 1
MAX_TRAIN_SAMPLES = 10000
TEMPERATURE_ENS  = 0.2
best_val_pck    = -1.0
history_ens     = []

for epoch in range(1, TRAIN_EPOCHS + 1):
    weight_module.train()
    total_loss = 0.0
    n_steps    = 0

    for idx, sample in enumerate(train_ds):
        if idx >= MAX_TRAIN_SAMPLES:
            break
        src_imgs = sample['src_img'].unsqueeze(0).to(device)
        tgt_imgs = sample['trg_img'].unsqueeze(0).to(device)
        src_sz   = (sample['src_imsize'][2], sample['src_imsize'][1])
        tgt_sz   = (sample['trg_imsize'][2], sample['trg_imsize'][1])
        src_kps  = sample['src_kps'].numpy()
        tgt_kps  = sample['trg_kps'].numpy()

        # Extract and cache features once per pair (no gradients needed for model weights)
        feat_cache = []
        with torch.no_grad():
            for mdl, rs, ps, is_sam in models_info:
                src_r = F.interpolate(src_imgs, size=(rs, rs), mode='bilinear', align_corners=False)
                tgt_r = F.interpolate(tgt_imgs, size=(rs, rs), mode='bilinear', align_corners=False)
                if is_sam:
                    sf = extract_dense_features_SAM(mdl, src_r, image_size=rs)
                    tf = extract_dense_features_SAM(mdl, tgt_r, image_size=rs)
                else:
                    sf = extract_dense_features(mdl, src_r)
                    tf = extract_dense_features(mdl, tgt_r)
                _, H, W, D = tf.shape
                feat_cache.append((sf.detach(), tf.reshape(H * W, D).detach(), H, W))

        # Compute loss — only weight_module gets gradients
        w = weight_module()
        sample_loss = torch.tensor(0.0, device=device)
        count = 0

        for ki in range(src_kps.shape[0]):
            rs0, ps0 = IMG_SIZES[0], PATCH_SIZES[0]
            spx, spy = pixel_to_patch_coord(src_kps[ki,0], src_kps[ki,1], src_sz, ps0, rs0)
            tpx, tpy = pixel_to_patch_coord(tgt_kps[ki,0], tgt_kps[ki,1], tgt_sz, ps0, rs0)
            grid_H   = rs0 // ps0
            gt_x     = tpx * SAM_GRID_SIZE // grid_H
            gt_y     = tpy * SAM_GRID_SIZE // grid_H
            gt_idx_sam = gt_y * SAM_GRID_SIZE + gt_x

            fused_sim = None
            for mi, (mdl, rs, ps, is_sam) in enumerate(models_info):
                sf, tf_flat, H, W = feat_cache[mi]
                src_px_m, src_py_m = pixel_to_patch_coord(src_kps[ki,0], src_kps[ki,1], src_sz, ps, rs)
                sim = F.cosine_similarity(sf[0, src_py_m, src_px_m, :].unsqueeze(0), tf_flat, dim=1)
                sim_2d = sim.reshape(1, 1, H, W)
                sim_rs = F.interpolate(sim_2d, size=(SAM_GRID_SIZE, SAM_GRID_SIZE),
                                       mode='bilinear', align_corners=False).reshape(-1)
                fused_sim = w[mi] * sim_rs if fused_sim is None else fused_sim + w[mi] * sim_rs

            logits   = fused_sim * TEMPERATURE_ENS
            loss_kp  = F.cross_entropy(logits.unsqueeze(0),
                                        torch.tensor([gt_idx_sam], device=device))
            sample_loss = sample_loss + loss_kp
            count += 1

        if count > 0:
            sample_loss = sample_loss / count
            optimizer_w.zero_grad()
            sample_loss.backward()
            optimizer_w.step()
            total_loss += sample_loss.item()
            n_steps    += 1

        if (idx + 1) % 500 == 0:
            print(f'  Epoch {epoch} step {idx+1}: avg_loss={total_loss/max(n_steps,1):.4f}')

    w_val = weight_module().detach().cpu().numpy().tolist()
    print(f'Epoch {epoch} done. Weights: {[f"{x:.4f}" for x in w_val]}')
    history_ens.append({'epoch': epoch, 'loss': total_loss/max(n_steps,1), 'weights': w_val})

w_final = weight_module().detach().cpu().numpy().tolist()
w_path  = os.path.join(RESULTS_DIR, 'learned_weights.json')
with open(w_path, 'w') as f:
    json.dump({'weights': w_final, 'history': history_ens}, f, indent=2)
print(f'Learned weights saved: {w_final}')
print(f'Saved to {w_path}')

/tmp/ipykernel_1779/3325715967.py:59: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


  Epoch 1 step 500: avg_loss=6.8085
  Epoch 1 step 1000: avg_loss=6.8041
  Epoch 1 step 1500: avg_loss=6.8007
  Epoch 1 step 2000: avg_loss=6.7978
  Epoch 1 step 2500: avg_loss=6.7966
  Epoch 1 step 3000: avg_loss=6.7964
  Epoch 1 step 3500: avg_loss=6.7991
  Epoch 1 step 4000: avg_loss=6.8007
  Epoch 1 step 4500: avg_loss=6.8021
  Epoch 1 step 5000: avg_loss=6.8030
  Epoch 1 step 5500: avg_loss=6.8040
  Epoch 1 step 6000: avg_loss=6.8047
  Epoch 1 step 6500: avg_loss=6.8047
  Epoch 1 step 7000: avg_loss=6.8049
  Epoch 1 step 7500: avg_loss=6.8050
  Epoch 1 step 8000: avg_loss=6.8050
  Epoch 1 step 8500: avg_loss=6.8051
  Epoch 1 step 9000: avg_loss=6.8051
  Epoch 1 step 9500: avg_loss=6.8044
  Epoch 1 step 10000: avg_loss=6.8039
Epoch 1 done. Weights: ['1.0000', '0.0000', '0.0000']
Learned weights saved: [0.9999845027923584, 7.492847998946672e-06, 7.94960851635551e-06]
Saved to /content/drive/MyDrive/semantic_correspondence/results/step4/ensemble/learned_weights.json


In [17]:
# Cell 6 — Ensemble evaluation function + SPair-71k test
def evaluate_ensemble(models_info, weights, dataset, device, thresholds, grid_size=32):
    """Fused ensemble evaluation. Features cached once per pair."""
    per_img = []
    with torch.no_grad():
        for idx, sample in enumerate(dataset):
            src_imgs = sample['src_img'].unsqueeze(0).to(device)
            tgt_imgs = sample['trg_img'].unsqueeze(0).to(device)
            src_sz   = (sample['src_imsize'][2], sample['src_imsize'][1])
            tgt_sz   = (sample['trg_imsize'][2], sample['trg_imsize'][1])
            rs_ref   = models_info[0][1]
            ps_ref   = models_info[0][2]
            src_kps  = sample['src_kps'].numpy()
            trg_kps  = sample['trg_kps'].numpy()
            trg_bbox = sample['trg_bbox']

            # Cache features once per pair
            feat_cache = []
            for mdl, rs, ps, is_sam in models_info:
                src_r = F.interpolate(src_imgs, size=(rs, rs), mode='bilinear', align_corners=False)
                tgt_r = F.interpolate(tgt_imgs, size=(rs, rs), mode='bilinear', align_corners=False)
                if is_sam:
                    sf = extract_dense_features_SAM(mdl, src_r, image_size=rs)
                    tf = extract_dense_features_SAM(mdl, tgt_r, image_size=rs)
                else:
                    sf = extract_dense_features(mdl, src_r)
                    tf = extract_dense_features(mdl, tgt_r)
                _, H, W, D = tf.shape
                feat_cache.append((sf, tf.reshape(H * W, D), H, W))

            preds = []
            for ki in range(src_kps.shape[0]):
                fused_sim = None
                for mi, (mdl, rs, ps, is_sam) in enumerate(models_info):
                    sf, tf_flat, H, W = feat_cache[mi]
                    src_px_m, src_py_m = pixel_to_patch_coord(src_kps[ki,0], src_kps[ki,1], src_sz, ps, rs)
                    sim    = F.cosine_similarity(sf[0, src_py_m, src_px_m, :].unsqueeze(0), tf_flat, dim=1)
                    sim_2d = sim.reshape(1, 1, H, W)
                    sim_rs = F.interpolate(sim_2d, size=(grid_size, grid_size),
                                           mode='bilinear', align_corners=False).reshape(-1)
                    fused_sim = weights[mi] * sim_rs if fused_sim is None else fused_sim + weights[mi] * sim_rs

                best_idx = fused_sim.argmax().item()
                gx = best_idx % grid_size
                gy = best_idx // grid_size
                ref_grid = rs_ref // ps_ref
                patch_x  = gx * ref_grid // grid_size
                patch_y  = gy * ref_grid // grid_size
                rx, ry   = patch_to_pixel_coord(patch_x, patch_y, tgt_sz, ps_ref, rs_ref)
                preds.append([rx, ry])

            pcks = {}
            for thr in thresholds:
                pck, _, _ = compute_pck_spair71k(preds, trg_kps.tolist(), trg_bbox, thr)
                pcks[thr] = pck
            per_img.append({'category': sample['category'], 'pck_scores': pcks})
            if (idx + 1) % 200 == 0:
                print(f'  {idx+1} pairs...')
    return per_img

test_ds     = SPairDataset(PAIR_ANN_PATH, LAYOUT_PATH, IMAGE_PATH, DATASET_SIZE, PCK_ALPHA, datatype='test')
models_info = [
    (dinov2, IMG_SIZES[0], PATCH_SIZES[0], False),
    (model2, IMG_SIZES[1], PATCH_SIZES[1], False),
    (sam,    512, 16, True),
]

print(f'=== Ensemble (learned weights={[f"{x:.4f}" for x in w_final]}) ===')
res_ens_learned = evaluate_ensemble(models_info, w_final, test_ds, device, THRESHOLDS)
stats_learned   = save_exp_results(res_ens_learned, 'ensemble_learned', THRESHOLDS, RESULTS_DIR)

fixed_w = [0.25, 0.65, 0.10]
print(f'\n=== Ensemble (fixed weights={fixed_w}) ===')
res_ens_fixed = evaluate_ensemble(models_info, fixed_w, test_ds, device, THRESHOLDS)
stats_fixed   = save_exp_results(res_ens_fixed, 'ensemble_fixed', THRESHOLDS, RESULTS_DIR)

=== Ensemble (learned weights=['1.0000', '0.0000', '0.0000']) ===


/tmp/ipykernel_1779/3325715967.py:59: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


  200 pairs...
  400 pairs...
  600 pairs...
  800 pairs...
  1000 pairs...
  1200 pairs...
  1400 pairs...
  1600 pairs...
  1800 pairs...
  2000 pairs...
  2200 pairs...
  2400 pairs...
  2600 pairs...
  2800 pairs...
  3000 pairs...
  3200 pairs...
  3400 pairs...
  3600 pairs...
  3800 pairs...
  4000 pairs...
  4200 pairs...
  4400 pairs...
  4600 pairs...
  4800 pairs...
  5000 pairs...
  5200 pairs...
  5400 pairs...
  5600 pairs...
  5800 pairs...
  6000 pairs...
  6200 pairs...
  6400 pairs...
  6600 pairs...
  6800 pairs...
  7000 pairs...
  7200 pairs...
  7400 pairs...
  7600 pairs...
  7800 pairs...
  8000 pairs...
  8200 pairs...
  8400 pairs...
  8600 pairs...
  8800 pairs...
  9000 pairs...
  9200 pairs...
  9400 pairs...
  9600 pairs...
  9800 pairs...
  10000 pairs...
  10200 pairs...
  10400 pairs...
  10600 pairs...
  10800 pairs...
  11000 pairs...
  11200 pairs...
  11400 pairs...
  11600 pairs...
  11800 pairs...
  12000 pairs...
  12200 pairs...
  PCK@0.05: 38.6

In [18]:
# Cell 7 — Individual model baselines for comparison
def evaluate_single(model, dataset, device, thresholds, img_size, patch_size, is_sam=False):
    per_img = []
    model.eval()
    with torch.no_grad():
        for idx, sample in enumerate(dataset):
            src_t = F.interpolate(sample['src_img'].unsqueeze(0).to(device),
                                   size=(img_size, img_size), mode='bilinear', align_corners=False)
            tgt_t = F.interpolate(sample['trg_img'].unsqueeze(0).to(device),
                                   size=(img_size, img_size), mode='bilinear', align_corners=False)
            src_sz = (sample['src_imsize'][2], sample['src_imsize'][1])
            tgt_sz = (sample['trg_imsize'][2], sample['trg_imsize'][1])
            if is_sam:
                sf = extract_dense_features_SAM(model, src_t, image_size=img_size)
                tf = extract_dense_features_SAM(model, tgt_t, image_size=img_size)
            else:
                sf = extract_dense_features(model, src_t)
                tf = extract_dense_features(model, tgt_t)
            _, H, W, D = tf.shape
            tf_flat = tf.reshape(H * W, D)
            src_kps = sample['src_kps'].numpy()
            trg_kps = sample['trg_kps'].numpy()
            trg_bbox = sample['trg_bbox']
            preds = []
            for i in range(src_kps.shape[0]):
                px, py = pixel_to_patch_coord(src_kps[i,0], src_kps[i,1], src_sz, patch_size, img_size)
                sim = F.cosine_similarity(sf[0, py, px, :].unsqueeze(0), tf_flat, dim=1)
                mx, my = find_best_match_argmax(sim, W)
                rx, ry = patch_to_pixel_coord(mx, my, tgt_sz, patch_size, img_size)
                preds.append([rx, ry])
            pcks = {}
            for thr in thresholds:
                pck, _, _ = compute_pck_spair71k(preds, trg_kps.tolist(), trg_bbox, thr)
                pcks[thr] = pck
            per_img.append({'category': sample['category'], 'pck_scores': pcks})
            if (idx + 1) % 200 == 0:
                print(f'  {idx+1} pairs...')
    return per_img

print('=== DINOv2 alone ===')
res_d2 = evaluate_single(dinov2, test_ds, device, THRESHOLDS, IMG_SIZES[0], PATCH_SIZES[0])
stats_d2 = save_exp_results(res_d2, 'ensemble_dinov2_alone', THRESHOLDS, RESULTS_DIR)

if DINOV3_OK:
    print('\n=== DINOv3 alone ===')
    res_d3 = evaluate_single(dinov3, test_ds, device, THRESHOLDS, IMG_SIZES[1], PATCH_SIZES[1])
    stats_d3 = save_exp_results(res_d3, 'ensemble_dinov3_alone', THRESHOLDS, RESULTS_DIR)
else:
    stats_d3 = None

print('\n=== SAM alone ===')
res_sam = evaluate_single(sam, test_ds, device, THRESHOLDS, 512, 16, is_sam=True)
stats_sam = save_exp_results(res_sam, 'ensemble_sam_alone', THRESHOLDS, RESULTS_DIR)

=== DINOv2 alone ===
  200 pairs...
  400 pairs...
  600 pairs...
  800 pairs...
  1000 pairs...
  1200 pairs...
  1400 pairs...
  1600 pairs...
  1800 pairs...
  2000 pairs...
  2200 pairs...
  2400 pairs...
  2600 pairs...
  2800 pairs...
  3000 pairs...
  3200 pairs...
  3400 pairs...
  3600 pairs...
  3800 pairs...
  4000 pairs...
  4200 pairs...
  4400 pairs...
  4600 pairs...
  4800 pairs...
  5000 pairs...
  5200 pairs...
  5400 pairs...
  5600 pairs...
  5800 pairs...
  6000 pairs...
  6200 pairs...
  6400 pairs...
  6600 pairs...
  6800 pairs...
  7000 pairs...
  7200 pairs...
  7400 pairs...
  7600 pairs...
  7800 pairs...
  8000 pairs...
  8200 pairs...
  8400 pairs...
  8600 pairs...
  8800 pairs...
  9000 pairs...
  9200 pairs...
  9400 pairs...
  9600 pairs...
  9800 pairs...
  10000 pairs...
  10200 pairs...
  10400 pairs...
  10600 pairs...
  10800 pairs...
  11000 pairs...
  11200 pairs...
  11400 pairs...
  11600 pairs...
  11800 pairs...
  12000 pairs...
  12200 pair

/tmp/ipykernel_1779/3325715967.py:59: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


  200 pairs...
  400 pairs...
  600 pairs...
  800 pairs...
  1000 pairs...
  1200 pairs...
  1400 pairs...
  1600 pairs...
  1800 pairs...
  2000 pairs...
  2200 pairs...
  2400 pairs...
  2600 pairs...
  2800 pairs...
  3000 pairs...
  3200 pairs...
  3400 pairs...
  3600 pairs...
  3800 pairs...
  4000 pairs...
  4200 pairs...
  4400 pairs...
  4600 pairs...
  4800 pairs...
  5000 pairs...
  5200 pairs...
  5400 pairs...
  5600 pairs...
  5800 pairs...
  6000 pairs...
  6200 pairs...
  6400 pairs...
  6600 pairs...
  6800 pairs...
  7000 pairs...
  7200 pairs...
  7400 pairs...
  7600 pairs...
  7800 pairs...
  8000 pairs...
  8200 pairs...
  8400 pairs...
  8600 pairs...
  8800 pairs...
  9000 pairs...
  9200 pairs...
  9400 pairs...
  9600 pairs...
  9800 pairs...
  10000 pairs...
  10200 pairs...
  10400 pairs...
  10600 pairs...
  10800 pairs...
  11000 pairs...
  11200 pairs...
  11400 pairs...
  11600 pairs...
  11800 pairs...
  12000 pairs...
  12200 pairs...
  PCK@0.05: 11.7

In [19]:
# Cell 8 — Summary table
rows = [
    {'Method': 'DINOv2 alone',        'Stats': stats_d2},
    {'Method': 'DINOv3 alone',        'Stats': stats_d3},
    {'Method': 'SAM alone',           'Stats': stats_sam},
    {'Method': 'Ensemble (fixed)',    'Stats': stats_fixed},
    {'Method': 'Ensemble (learned)',  'Stats': stats_learned},
]
table = []
for r in rows:
    if r['Stats'] is None:
        table.append({'Method': r['Method'], 'PCK@0.05': '-', 'PCK@0.10': '-', 'PCK@0.20': '-'})
    else:
        table.append({
            'Method': r['Method'],
            'PCK@0.05': f"{r['Stats'].get('pck@0.05',{}).get('mean',0):.2f}%",
            'PCK@0.10': f"{r['Stats'].get('pck@0.10',{}).get('mean',0):.2f}%",
            'PCK@0.20': f"{r['Stats'].get('pck@0.20',{}).get('mean',0):.2f}%",
        })
df = pd.DataFrame(table)
print('\n=== Step 4c Results: Ensemble Comparison ===')
print(df.to_string(index=False))
print(f'\nLearned weights: DINOv2={w_final[0]:.4f}, DINOv3={w_final[1]:.4f}, SAM={w_final[2]:.4f}')
df.to_csv(f'{RESULTS_DIR}/step4c_summary.csv', index=False)
print(f'Saved to {RESULTS_DIR}/step4c_summary.csv')


=== Step 4c Results: Ensemble Comparison ===
            Method PCK@0.05 PCK@0.10 PCK@0.20
      DINOv2 alone   53.13%   67.42%   78.79%
      DINOv3 alone   32.38%   48.27%   62.83%
         SAM alone   11.72%   20.45%   33.39%
  Ensemble (fixed)   39.07%   60.67%   74.93%
Ensemble (learned)   38.64%   60.08%   74.49%

Learned weights: DINOv2=1.0000, DINOv3=0.0000, SAM=0.0000
Saved to /content/drive/MyDrive/semantic_correspondence/results/step4/ensemble/step4c_summary.csv
